In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

# Add your code files to Python path
sys.path.insert(0, "/kaggle/input/datasets/yosrkharrat/deepfake/")
# Load the manifest
manifest_path = "/kaggle/input/datasets/raedsaidi/deepfake/processed/precompute_manifest.csv"  # ← fixed (removed FaceForensics++_C23 subfolder)
df = pd.read_csv(manifest_path)


print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nFirst path:", df['path'].iloc[0])
print("\nLabel counts:\n", df['label'].value_counts())

In [ ]:
# The paths in the CSV were written on another machine
# We need to remap them to where Kaggle mounted the dataset

KAGGLE_PROCESSED = "/kaggle/input/datasets/raedsaidi/deepfake/processed"

def fix_path(old_path):
    p = Path(old_path)
    # Find the part starting from FaceForensics++_C23
    parts = p.parts
    try:
        idx = next(i for i, part in enumerate(parts) if "FaceForensics" in part)
        relative = Path(*parts[idx:])
        return str(Path(KAGGLE_PROCESSED) / relative)
    except StopIteration:
        return old_path  # already relative or unknown format

df['path'] = df['path'].apply(fix_path)
df['label'] = df['label'].astype(int)

# Verify fix
print("Fixed path:", df['path'].iloc[0])
print("File exists?", Path(df['path'].iloc[0]).exists())
print("\nLabel counts:\n", df['label'].value_counts())
# Save fixed manifest
df.to_csv("/kaggle/working/fixed_manifest.csv", index=False)
print("Saved fixed manifest.")

In [ ]:
import os
from pathlib import Path

base = "/kaggle/input/datasets/raedsaidi/deepfake/processed/FaceForensics++_C23"

fake_imgs = list(Path(base + "/fake").rglob("*.jpg"))
real_imgs = list(Path(base + "/real").rglob("*.jpg"))

print(f"Fake images found: {len(fake_imgs)}")
print(f"Real images found: {len(real_imgs)}")
print("\nSample fake:", fake_imgs[0] if fake_imgs else "NONE")
print("Sample real:", real_imgs[0] if real_imgs else "NONE")

In [ ]:
records = []

for img_path in Path(base + "/fake").rglob("*.jpg"):
    records.append({"path": str(img_path), "label": 1, "source_dataset": "FaceForensics++_C23"})

for img_path in Path(base + "/real").rglob("*.jpg"):
    records.append({"path": str(img_path), "label": 0, "source_dataset": "FaceForensics++_C23"})

df = pd.DataFrame(records)
print("New label counts:\n", df['label'].value_counts())
print("Total:", len(df))

df.to_csv("/kaggle/working/fixed_manifest.csv", index=False)
print("Saved rebuilt manifest.")

In [ ]:
!pip install albumentations -q

In [ ]:
labels_all = df['label'].tolist()  # if df is in scope, or read from CSV:

# safer — read directly from the rebuilt manifest
df_check = pd.read_csv("/kaggle/working/fixed_manifest.csv")
real_count = (df_check['label'] == 0).sum()
fake_count = (df_check['label'] == 1).sum()
print(f"Real: {real_count} | Fake: {fake_count}")

In [ ]:
import sys
import types
import importlib.util

sys.path.insert(0, "/kaggle/input/datasets/yosrkharrat/deepfake/")

BASE = "/kaggle/input/datasets/yosrkharrat/deepfake"

def patch_module(full_name, file_path):
    spec = importlib.util.spec_from_file_location(full_name, file_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    spec.loader.exec_module(mod)
    return mod

# Create fake parent packages
for pkg in ["src", "src.models", "src.utils"]:
    if pkg not in sys.modules:
        sys.modules[pkg] = types.ModuleType(pkg)

# Patch the actual .py files into the fake packages
patch_module("src.models.fft_stream",    f"{BASE}/fft_stream.py")
patch_module("src.utils.fft_transform",  f"{BASE}/fft_transform.py")

# Also alias them at top level (in case other files import directly)
sys.modules["fft_stream"]    = sys.modules["src.models.fft_stream"]
sys.modules["fft_transform"] = sys.modules["src.utils.fft_transform"]

print("Patched src.models.fft_stream ✅")
print("Patched src.utils.fft_transform ✅")

In [ ]:
import sys, types, importlib.util

BASE = "/kaggle/input/datasets/yosrkharrat/deepfake"
sys.path.insert(0, BASE)

# Build the full fake src.* package tree
for pkg in ["src", "src.data", "src.models", "src.training", "src.utils"]:
    sys.modules[pkg] = types.ModuleType(pkg)

def patch(full_name, filename):
    spec = importlib.util.spec_from_file_location(full_name, f"{BASE}/{filename}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    # Also register short name for direct imports
    sys.modules[filename.replace('.py','')] = mod
    spec.loader.exec_module(mod)
    return mod

# Order matters — fft_stream must come before fft_transform and dataset
patch("src.models.fft_stream",    "fft_stream.py")
patch("src.utils.fft_transform",  "fft_transform.py")
patch("src.utils.metrics",        "metrics.py")
patch("src.data.augmentation",    "augmentation.py")
patch("src.data.dataset",         "dataset.py")
patch("src.training.losses",      "losses.py")

print("All modules patched ✅")

# ── Now run training ──────────────────────────────────────────────────────
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import pandas as pd

from src.models.fft_stream import FFTOnlyClassifier
from src.data.dataset import DeepfakeDataset
from src.data.augmentation import get_train_augmentation, get_eval_augmentation
from src.training.losses import get_loss
from src.utils.metrics import compute_metrics

CSV_PATH    = "/kaggle/working/fixed_manifest.csv"
EPOCHS      = 15
BATCH_SIZE  = 32
LR          = 1e-3      # increased from 1e-4 — model was too slow to learn
VAL_RATIO   = 0.15
NUM_WORKERS = 2
SAVE_PATH   = "/kaggle/working/fft_model.pth"
SEED        = 42

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

full_ds  = DeepfakeDataset(CSV_PATH, transform=get_train_augmentation(), use_fft=False)
val_size = int(VAL_RATIO * len(full_ds))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size],
                                 generator=torch.Generator().manual_seed(SEED))

val_base_ds = DeepfakeDataset(CSV_PATH, transform=get_eval_augmentation(), use_fft=False)
val_ds.dataset = val_base_ds

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

labels_all = [int(rec["label"]) for rec in full_ds.records]
real_count = labels_all.count(0)
fake_count = labels_all.count(1)
print(f"Real: {real_count} | Fake: {fake_count}")

model     = FFTOnlyClassifier(dropout=0.3).to(device)
criterion = get_loss(real_count, fake_count).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(model, loader, criterion, optimizer, training):
    model.train() if training else model.eval()
    total_loss, all_labels, all_preds, all_probs = 0.0, [], [], []
    with torch.set_grad_enabled(training):
        for batch in loader:
            rgb, labels = batch[0].to(device), batch[-1].to(device)
            if training:
                optimizer.zero_grad()
            logits = model(rgb)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            probs       = torch.softmax(logits, dim=1)[:, 1]
            total_loss  += loss.item()
            all_preds   += logits.argmax(1).detach().cpu().tolist()
            all_labels  += labels.detach().cpu().tolist()
            all_probs   += probs.detach().cpu().tolist()
    m = compute_metrics(all_labels, all_preds, all_probs)
    m["loss"] = total_loss / max(len(loader), 1)
    return m

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_m   = run_epoch(model, val_loader,   criterion, optimizer, training=False)
    scheduler.step()
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.3f} | "
        f"Val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.3f} "
        f"F1 {val_m['f1']:.3f} AUC {val_m['auc_roc']:.3f}"
    )
    if val_m["auc_roc"] > best_auc:
        best_auc = val_m["auc_roc"]
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model (AUC {best_auc:.3f})")

print(f"\nDone. Best val AUC: {best_auc:.3f}")

In [ ]:
import torch
import numpy as np

# ── 1. Check if gradients are actually flowing ─────────────────────────
model.train()
batch = next(iter(train_loader))
rgb, labels = batch[0].to(device), batch[-1].to(device)

print("=== Batch info ===")
print(f"RGB shape: {rgb.shape}, dtype: {rgb.dtype}")
print(f"RGB range: [{rgb.min():.3f}, {rgb.max():.3f}]")
print(f"Labels: {labels[:16].tolist()}")
print(f"Label unique: {labels.unique()}")

optimizer.zero_grad()
logits = model(rgb)
loss = criterion(logits, labels)
loss.backward()

print("\n=== Gradient check ===")
for name, p in model.named_parameters():
    if p.grad is not None:
        print(f"{name:50s} grad norm: {p.grad.norm():.6f}")
    else:
        print(f"{name:50s} NO GRAD")

# ── 2. Check logits ────────────────────────────────────────────────────
print("\n=== Logits (first 8) ===")
print(logits[:8].detach().cpu())
print("Softmax probs:", torch.softmax(logits[:8], dim=1).detach().cpu())

# ── 3. Check FFT output ────────────────────────────────────────────────
from src.models.fft_stream import compute_fft_magnitude
fft_out = compute_fft_magnitude(rgb[:4])
print(f"\n=== FFT output ===")
print(f"Shape: {fft_out.shape}")
print(f"Range: [{fft_out.min():.4f}, {fft_out.max():.4f}]")
print(f"Mean: {fft_out.mean():.4f}, Std: {fft_out.std():.4f}")
print(f"Any NaN: {fft_out.isnan().any()}, Any Inf: {fft_out.isinf().any()}")

# ── 4. Check if real and fake FFTs are actually different ──────────────
real_idx = (labels == 0).nonzero(as_tuple=True)[0][:4]
fake_idx = (labels == 1).nonzero(as_tuple=True)[0][:4]

if len(real_idx) > 0 and len(fake_idx) > 0:
    real_fft = compute_fft_magnitude(rgb[real_idx])
    fake_fft = compute_fft_magnitude(rgb[fake_idx])
    print(f"\n=== Real vs Fake FFT ===")
    print(f"Real FFT mean: {real_fft.mean():.4f}, std: {real_fft.std():.4f}")
    print(f"Fake FFT mean: {fake_fft.mean():.4f}, std: {fake_fft.std():.4f}")
    diff = (real_fft.mean() - fake_fft.mean()).abs()
    print(f"Mean difference: {diff:.6f}")
else:
    print("\n⚠️  Batch has only one class — imbalance is so severe single batches are mono-class")
    print(f"Real in batch: {(labels==0).sum()}, Fake in batch: {(labels==1).sum()}")

In [ ]:
import sys, types, importlib.util, torch, torch.nn as nn, torch.nn.functional as F

BASE = "/kaggle/input/datasets/yosrkharrat/deepfake"

# ── Patched fft_stream with FIXED compute_fft_magnitude ──────────────────
FIXED_FFT_STREAM = '''
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_fft_magnitude(image_tensor: torch.Tensor) -> torch.Tensor:
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)
    if image_tensor.shape[1] == 3:
        gray = (0.2989 * image_tensor[:, 0]
              + 0.5870 * image_tensor[:, 1]
              + 0.1140 * image_tensor[:, 2]).unsqueeze(1)
    else:
        gray = image_tensor

    # Use rfft2 — real FFT, fully differentiable, no fftshift needed
    fft = torch.fft.rfft2(gray, norm="ortho")
    magnitude = torch.log1p(fft.abs())

    # Normalize per image
    b = magnitude.shape[0]
    mag_flat = magnitude.view(b, -1)
    mag_min = mag_flat.min(dim=1).values.view(b, 1, 1, 1)
    mag_max = mag_flat.max(dim=1).values.view(b, 1, 1, 1)
    magnitude = (magnitude - mag_min) / (mag_max - mag_min + 1e-8)
    return magnitude  # (B, 1, H, W//2+1)


class FFTBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class FFTStreamCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        # rfft2 output width is W//2+1 = 113, so use adaptive pooling — no shape issues
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.block1 = FFTBlock(32,  64,  stride=2)
        self.block2 = FFTBlock(64,  128, stride=2)
        self.block3 = FFTBlock(128, 256, stride=2)
        self.block4 = FFTBlock(256, 256, stride=2)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(256, 256)

    def forward(self, x):
        if x.shape[1] == 3:
            x = compute_fft_magnitude(x)
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.head(x)


class FFTOnlyClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone = FFTStreamCNN(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))
'''

# ── Register patched modules ──────────────────────────────────────────────
import sys, types, importlib.util

for pkg in ["src", "src.data", "src.models", "src.training", "src.utils"]:
    sys.modules[pkg] = types.ModuleType(pkg)

def patch_file(full_name, filename):
    spec = importlib.util.spec_from_file_location(full_name, f"{BASE}/{filename}")
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    sys.modules[filename.replace('.py','')] = mod
    spec.loader.exec_module(mod)
    return mod

# Patch fft_stream from string (fixed version)
fft_mod = types.ModuleType("src.models.fft_stream")
exec(FIXED_FFT_STREAM, fft_mod.__dict__)
sys.modules["src.models.fft_stream"] = fft_mod
sys.modules["fft_stream"] = fft_mod

# Patch rest from files
patch_file("src.utils.fft_transform", "fft_transform.py")
patch_file("src.utils.metrics",       "metrics.py")
patch_file("src.data.augmentation",   "augmentation.py")
patch_file("src.data.dataset",        "dataset.py")
patch_file("src.training.losses",     "losses.py")

print("All modules patched ✅")

# ── Verify gradients flow before full training ────────────────────────────
from src.models.fft_stream import FFTOnlyClassifier, compute_fft_magnitude
from src.training.losses import get_loss

test_model = FFTOnlyClassifier(dropout=0.3).to(device)
test_input = torch.randn(4, 3, 224, 224).to(device)
test_labels = torch.tensor([0, 1, 0, 1]).to(device)
test_loss = get_loss(10499, 22399).to(device)

out = test_model(test_input)
loss = test_loss(out, test_labels)
loss.backward()

grad_norms = [p.grad.norm().item() for p in test_model.parameters() if p.grad is not None]
print(f"Layers with gradients: {len(grad_norms)}/{len(list(test_model.parameters()))}")
print(f"Min grad norm: {min(grad_norms):.6f}")
print(f"Max grad norm: {max(grad_norms):.6f}")
print(f"Logits: {out.detach().cpu()}")
print("Gradient flow: ✅" if min(grad_norms) > 1e-8 else "❌ Still dead")

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from src.models.fft_stream import FFTOnlyClassifier
from src.data.dataset import DeepfakeDataset
from src.data.augmentation import get_train_augmentation, get_eval_augmentation
from src.training.losses import get_loss
from src.utils.metrics import compute_metrics

CSV_PATH    = "/kaggle/working/fixed_manifest.csv"
EPOCHS      = 15
BATCH_SIZE  = 32
LR          = 1e-3
VAL_RATIO   = 0.15
NUM_WORKERS = 2
SAVE_PATH   = "/kaggle/working/fft_model.pth"
SEED        = 42

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

full_ds  = DeepfakeDataset(CSV_PATH, transform=get_train_augmentation(), use_fft=False)
val_size = int(VAL_RATIO * len(full_ds))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size],
                                 generator=torch.Generator().manual_seed(SEED))

val_base_ds = DeepfakeDataset(CSV_PATH, transform=get_eval_augmentation(), use_fft=False)
val_ds.dataset = val_base_ds

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

labels_all = [int(rec["label"]) for rec in full_ds.records]
real_count = labels_all.count(0)
fake_count = labels_all.count(1)
print(f"Real: {real_count} | Fake: {fake_count}")

model     = FFTOnlyClassifier(dropout=0.3).to(device)
criterion = get_loss(real_count, fake_count).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(model, loader, criterion, optimizer, training):
    model.train() if training else model.eval()
    total_loss, all_labels, all_preds, all_probs = 0.0, [], [], []
    with torch.set_grad_enabled(training):
        for batch in loader:
            rgb, labels = batch[0].to(device), batch[-1].to(device)
            if training:
                optimizer.zero_grad()
            logits = model(rgb)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            probs      = torch.softmax(logits, dim=1)[:, 1]
            total_loss += loss.item()
            all_preds  += logits.argmax(1).detach().cpu().tolist()
            all_labels += labels.detach().cpu().tolist()
            all_probs  += probs.detach().cpu().tolist()
    m = compute_metrics(all_labels, all_preds, all_probs)
    m["loss"] = total_loss / max(len(loader), 1)
    return m

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_m   = run_epoch(model, val_loader,   criterion, optimizer, training=False)
    scheduler.step()
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.3f} | "
        f"Val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.3f} "
        f"F1 {val_m['f1']:.3f} AUC {val_m['auc_roc']:.3f}"
    )
    if val_m["auc_roc"] > best_auc:
        best_auc = val_m["auc_roc"]
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model (AUC {best_auc:.3f})")

print(f"\nDone. Best val AUC: {best_auc:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# ── Run inference on full val set ─────────────────────────────────────
model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in val_loader:
        rgb, labels = batch[0].to(device), batch[-1].to(device)
        logits = model(rgb)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        all_preds  += logits.argmax(1).cpu().tolist()
        all_labels += labels.cpu().tolist()
        all_probs  += probs.cpu().tolist()

# ── Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: confusion matrix heatmap
ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Real (0)', 'Fake (1)'])
ax.set_yticklabels(['Real (0)', 'Fake (1)'])

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]}\n({cm[i,j]/cm[i].sum()*100:.1f}%)',
                ha='center', va='center', fontsize=13,
                color='white' if cm[i,j] > cm.max()/2 else 'black')

# Right: metrics summary
ax2 = axes[1]
ax2.axis('off')
total = len(all_labels)
accuracy  = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0

metrics_text = (
    f"{'Metric':<20} {'Value':>8}\n"
    f"{'─'*30}\n"
    f"{'AUC-ROC':<20} {'0.910':>8}\n"
    f"{'Accuracy':<20} {accuracy:>8.3f}\n"
    f"{'Precision':<20} {precision:>8.3f}\n"
    f"{'Recall (TPR)':<20} {recall:>8.3f}\n"
    f"{'F1 Score':<20} {f1:>8.3f}\n"
    f"{'FPR':<20} {fpr:>8.3f}\n"
    f"{'─'*30}\n"
    f"{'True Positives':<20} {tp:>8}\n"
    f"{'True Negatives':<20} {tn:>8}\n"
    f"{'False Positives':<20} {fp:>8}\n"
    f"{'False Negatives':<20} {fn:>8}\n"
    f"{'─'*30}\n"
    f"{'Total samples':<20} {total:>8}\n"
)
ax2.text(0.05, 0.95, metrics_text, transform=ax2.transAxes,
         fontsize=12, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
ax2.set_title('Evaluation Metrics', fontsize=14, fontweight='bold')

plt.suptitle('FFT-Stream Deepfake Detector — Validation Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to /kaggle/working/confusion_matrix.png")

In [ ]:
# ── Reload fresh model and train longer with better settings ─────────────
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from src.models.fft_stream import FFTOnlyClassifier
from src.data.dataset import DeepfakeDataset
from src.data.augmentation import get_train_augmentation, get_eval_augmentation
from src.training.losses import get_loss
from src.utils.metrics import compute_metrics

CSV_PATH    = "/kaggle/working/fixed_manifest.csv"
EPOCHS      = 40          # was 15
BATCH_SIZE  = 32
LR          = 1e-3
VAL_RATIO   = 0.15
NUM_WORKERS = 2
SAVE_PATH   = "/kaggle/working/fft_model_v2.pth"
SEED        = 42

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

full_ds  = DeepfakeDataset(CSV_PATH, transform=get_train_augmentation(), use_fft=False)
val_size = int(VAL_RATIO * len(full_ds))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size],
                                 generator=torch.Generator().manual_seed(SEED))
val_base_ds = DeepfakeDataset(CSV_PATH, transform=get_eval_augmentation(), use_fft=False)
val_ds.dataset = val_base_ds

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

labels_all = [int(rec["label"]) for rec in full_ds.records]
real_count = labels_all.count(0)
fake_count = labels_all.count(1)

model     = FFTOnlyClassifier(dropout=0.3).to(device)
criterion = get_loss(real_count, fake_count).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# OneCycleLR: warms up then anneals — much better than CosineAnnealing for longer runs
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,          # 10% warmup
    div_factor=10,          # start at lr/10
    final_div_factor=100,   # end at lr/100
)

def run_epoch(model, loader, criterion, optimizer, scheduler, training):
    model.train() if training else model.eval()
    total_loss, all_labels, all_preds, all_probs = 0.0, [], [], []
    with torch.set_grad_enabled(training):
        for batch in loader:
            rgb, labels = batch[0].to(device), batch[-1].to(device)
            if training:
                optimizer.zero_grad()
            logits = model(rgb)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # stability
                optimizer.step()
                scheduler.step()  # OneCycleLR steps per batch
            probs      = torch.softmax(logits, dim=1)[:, 1]
            total_loss += loss.item()
            all_preds  += logits.argmax(1).detach().cpu().tolist()
            all_labels += labels.detach().cpu().tolist()
            all_probs  += probs.detach().cpu().tolist()
    m = compute_metrics(all_labels, all_preds, all_probs)
    m["loss"] = total_loss / max(len(loader), 1)
    return m

best_auc  = 0.0
patience  = 8   # stop if no improvement for 8 epochs
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer, scheduler, training=True)
    val_m   = run_epoch(model, val_loader,   criterion, optimizer, scheduler, training=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.3f} | "
        f"Val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.3f} "
        f"F1 {val_m['f1']:.3f} AUC {val_m['auc_roc']:.3f} "
        f"Recall {val_m['recall']:.3f}"
    )

    if val_m["auc_roc"] > best_auc:
        best_auc   = val_m["auc_roc"]
        no_improve = 0

        
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model (AUC {best_auc:.3f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\n  Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

print(f"\nDone. Best val AUC: {best_auc:.3f}")

In [ ]:
# Resume from best checkpoint and train 40 more epochs
model.load_state_dict(torch.load(SAVE_PATH))
print("Loaded checkpoint ✅")

EPOCHS_EXTRA = 40
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,      # lower max_lr for fine-tuning
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_EXTRA,
    pct_start=0.05,
    div_factor=10,
    final_div_factor=1000,
)

best_auc   = 0.936  # continue from where we left off
no_improve = 0
SAVE_PATH2 = "/kaggle/working/fft_model_v3.pth"

for epoch in range(1, EPOCHS_EXTRA + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer, scheduler, training=True)
    val_m   = run_epoch(model, val_loader,   criterion, optimizer, scheduler, training=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS_EXTRA} | "
        f"Train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.3f} | "
        f"Val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.3f} "
        f"F1 {val_m['f1']:.3f} AUC {val_m['auc_roc']:.3f} "
        f"Recall {val_m['recall']:.3f}"
    )
    if val_m["auc_roc"] > best_auc:
        best_auc   = val_m["auc_roc"]
        no_improve = 0
        torch.save(model.state_dict(), SAVE_PATH2)
        print(f"  ✓ Saved best model (AUC {best_auc:.3f})")
    else:
        no_improve += 1
        if no_improve >= 10:
            print(f"\n  Early stopping at epoch {epoch}")
            break

print(f"\nDone. Best val AUC: {best_auc:.3f}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve

# Load best model and get probabilities
model.load_state_dict(torch.load(SAVE_PATH2 if best_auc > 0.936 else SAVE_PATH))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for batch in val_loader:
        rgb, labels = batch[0].to(device), batch[-1].to(device)
        logits = model(rgb)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        all_labels += labels.cpu().tolist()
        all_probs  += probs.cpu().tolist()

all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# Find threshold that maximizes F1
thresholds = np.arange(0.1, 0.9, 0.01)
best_f1, best_thresh = 0, 0.5
for t in thresholds:
    preds = (all_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
    f1 = 2*tp / (2*tp + fp + fn + 1e-8)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"Optimal threshold: {best_thresh:.2f}  (default was 0.50)")
print(f"\nResults at threshold {best_thresh:.2f}:")
preds = (all_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
total = len(all_labels)
print(f"  Accuracy:  {(tp+tn)/total:.3f}")
print(f"  Precision: {tp/(tp+fp):.3f}")
print(f"  Recall:    {tp/(tp+fn):.3f}")
print(f"  F1:        {best_f1:.3f}")
print(f"  FPR:       {fp/(fp+tn):.3f}")
print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

In [ ]:
import json
config = {"threshold": float(best_thresh), "auc": 0.936}
with open("/kaggle/working/inference_config.json", "w") as f:
    json.dump(config, f)
print("Saved inference config:", config)